# Unified IID Experiment — Score-Based Weak-Target Detection
**Single-class + Multi-class, one consistent protocol.** Additive target model only.

This notebook runs the full IID experiment end-to-end on Colab and extracts **each figure
separately** (for LaTeX assembly). It is **resumable**: re-running after a disconnect skips
already-trained models. All models, raw scores, ROC curves and metrics are saved.

**Pipeline**
1. Setup (Drive, deps, paths, device)
2. Configs — 4 runs: `single_n`, `single_rho`, `multi_n`, `multi_rho`
3. Train the 4 sweeps (AMF · DSM · LRao · GMM-Levin(+oracle))
4. Rescore → save raw scores/ROC/metrics → extract per-figure PDFs
5. Study A — DSM depends on preprocessing/scaling
6. Study B — LRao sensitivity (fast convergence · poor multiclass · degrades with n)
7. Figure gallery (each figure shown separately)
8. Package results to Drive

> The *only* deliberate difference between single- and multi-class is the score network:
> **linear** for the near-unimodal single class, **MLP [64,64]** for the multimodal mixture.
> Everything else (epochs, lr, normalisation, n-grid, rho-grid, seeds, metric) is identical.

## 1. Setup

In [ ]:
# Setup: mount Drive, clone the repo, set paths + device, place dataset.
import os, sys, subprocess

try:
    import google.colab          # noqa
    ON_COLAB = True
except Exception:
    ON_COLAB = False
print('Colab:', ON_COLAB)

# >>> EDIT THESE for your setup >>>
REPO_URL      = 'https://github.com/michaelpiro/final-paper-experiment.git'
REPO_BRANCH   = 'iid-unified-experiment'
# Drive paths (Colab): results persist here; the dataset is copied from here.
DRIVE_RESULTS = '/content/drive/MyDrive/final_paper_experiment/iid_results'
DRIVE_DATASET = '/content/drive/MyDrive/final_paper_experiment/pavia-u.mat'
# <<<

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/proj'
    if not os.path.isdir(os.path.join(PROJECT_DIR, '.git')):
        subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1',
                        REPO_URL, PROJECT_DIR], check=True)
    else:
        subprocess.run(['git', '-C', PROJECT_DIR, 'pull'], check=False)
    RESULTS_DIR = DRIVE_RESULTS
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                    'pymupdf', 'scikit-learn', 'pyyaml'], check=False)
else:
    PROJECT_DIR = os.path.abspath('.')
    RESULTS_DIR = os.path.join(PROJECT_DIR, 'experiments/honest_pipeline/results')

os.chdir(PROJECT_DIR); sys.path.insert(0, PROJECT_DIR)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Dataset: the repo ships only a (broken-on-Colab) symlink; copy the real .mat.
DATA = 'data/pavia-u.mat'
os.makedirs('data', exist_ok=True)
if not os.path.exists(DATA):
    if os.path.islink(DATA):
        os.remove(DATA)                       # drop broken symlink
    if ON_COLAB and os.path.exists(DRIVE_DATASET):
        import shutil; shutil.copy(DRIVE_DATASET, DATA)
        print('dataset copied from Drive')
assert os.path.exists(DATA), (
    f'pavia-u.mat missing. On Colab put it at {DRIVE_DATASET}; '
    f'locally put it at {PROJECT_DIR}/data/pavia-u.mat')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Project:', PROJECT_DIR)
print('Results:', RESULTS_DIR)
print('Device :', DEVICE, '| torch', torch.__version__)
print('Dataset:', DATA, 'OK')

## 2. Configs
The four configs live in `experiments/honest_pipeline/configs/iid/`. We patch `results_dir` to the (Drive) results path and keep the stable `run_name` so the run is resumable.

In [ ]:
import yaml, glob

CFG_SRC = 'experiments/honest_pipeline/configs/iid'
CFG_RUN = os.path.join(RESULTS_DIR, '_configs')
os.makedirs(CFG_RUN, exist_ok=True)

RUNS = {}   # name -> (config_path, run_dir)
for name in ['single_n', 'single_rho', 'multi_n', 'multi_rho']:
    cfg = yaml.safe_load(open(os.path.join(CFG_SRC, f'{name}.yaml')))
    cfg['results_dir'] = RESULTS_DIR
    cfg['dataset'] = DATA
    out = os.path.join(CFG_RUN, f'{name}.yaml')
    yaml.dump(cfg, open(out, 'w'), sort_keys=False)
    RUNS[name] = (out, os.path.join(RESULTS_DIR, cfg['run_name']))
    net = str(cfg['hidden_dims'] or 'linear')
    print(f"{name:10s}  net={net:>9}  bkg={cfg['bkg_cls']}  "
          f"n={cfg['n_train_list']}  |rho|={len(cfg['rho_list'])}  gmm={cfg.get('run_gmm_glrt', False)}")
print('\nrun dirs ->', {k: v[1] for k, v in RUNS.items()})

## 3. Train the four sweeps (resumable)
Each call trains AMF / DSM / LRao (and GMM-Levin + oracle for multiclass) over the swept axis, saving every model + pipeline. Re-running skips finished checkpoints.

In [ ]:
import subprocess, sys

def run_sweep(name):
    cfg_path, run_dir = RUNS[name]
    print(f"\n{'='*70}\n  SWEEP: {name}  ->  {run_dir}\n{'='*70}", flush=True)
    subprocess.run([sys.executable, '-u',
                    'experiments/honest_pipeline/run_sweep.py',
                    '--config', cfg_path], check=True)

for name in ['single_n', 'single_rho', 'multi_n', 'multi_rho']:
    run_sweep(name)
print('\nAll sweeps done.')

## 4. Rescore → raw scores + ROC + metrics → per-figure PDFs
`make_pauc_figures.py` reloads every model, reproduces the exact splits, recomputes ROC, dumps raw scores (`<run>/raw_scores/*.npz`) and emits each figure as its own PDF into `pauc_figures/`.

In [ ]:
FIG_DIR = os.path.join(RESULTS_DIR, 'pauc_figures')
subprocess.run([sys.executable, '-u',
                'experiments/honest_pipeline/make_pauc_figures.py',
                '--single_n',   RUNS['single_n'][1],
                '--single_rho', RUNS['single_rho'][1],
                '--multi_n',    RUNS['multi_n'][1],
                '--multi_rho',  RUNS['multi_rho'][1],
                '--out',        FIG_DIR], check=True)
print('Figures + caches ->', FIG_DIR)

In [ ]:
# Combined 2x2 paper figure (additive-only). Reads the caches just produced.
import os
os.environ['PAUC_CACHE_DIR'] = FIG_DIR    # plot_paper_2x2 reads from BASE/pauc_figures
subprocess.run([sys.executable, '-u',
                'experiments/honest_pipeline/plot_paper_2x2.py'], check=False)

## 5. Study A — DSM depends on preprocessing / scaling
Shows DSM fails on raw 103-D and on un-scaled PCA, but recovers with per-band standardisation.

In [ ]:
STUDY_DSM = os.path.join(RESULTS_DIR, 'study_dsm_preproc')
subprocess.run([sys.executable, '-u',
                'experiments/honest_pipeline/study_dsm_preprocessing.py',
                '--out', STUDY_DSM,
                '--seeds', '42', '43', '44', '45', '46'], check=True)

## 6. Study B — LRao sensitivity
(a) converges in a few epochs, (b) trails on multiclass, (c) AUC drops as n grows on multiclass.

In [ ]:
STUDY_LRAO = os.path.join(RESULTS_DIR, 'study_lrao')
subprocess.run([sys.executable, '-u',
                'experiments/honest_pipeline/study_lrao_sensitivity.py',
                '--out', STUDY_LRAO,
                '--single_cache', os.path.join(FIG_DIR, 'cache_single_n.pkl'),
                '--multi_cache',  os.path.join(FIG_DIR, 'cache_multi_n.pkl'),
                '--seeds', '42', '43', '44'], check=True)

## 7. Figure gallery — each figure separately
Every PDF is rendered inline with its filename caption, so it maps 1:1 to a LaTeX `\\includegraphics`.

In [ ]:
import glob, fitz   # PyMuPDF
from IPython.display import Image, display, Markdown

def show_pdf(path, dpi=130):
    doc = fitz.open(path)
    pix = doc[0].get_pixmap(dpi=dpi)
    png = path.replace('.pdf', '.png')
    pix.save(png)
    display(Markdown(f'**`{os.path.relpath(path, RESULTS_DIR)}`**'))
    display(Image(filename=png))

pdfs = []
for sub in ['pauc_figures', 'study_dsm_preproc', 'study_lrao']:
    pdfs += sorted(glob.glob(os.path.join(RESULTS_DIR, sub, '*.pdf')))
pdfs += sorted(glob.glob(os.path.join(RESULTS_DIR, 'paper_2x2.pdf')))
print(f'{len(pdfs)} figures:')
for p in pdfs:
    try:
        show_pdf(p)
    except Exception as e:
        print('  (could not render', p, ':', e, ')')

## 8. Package results to Drive
Zips models + raw scores + caches + metrics + figures for download / archival.

In [ ]:
import shutil
zip_base = os.path.join(RESULTS_DIR, 'iid_experiment_bundle')
# Bundle only the IID run dirs + figures + studies (skip unrelated old runs).
import tempfile, os
stage = tempfile.mkdtemp()
for name in ['single_n', 'single_rho', 'multi_n', 'multi_rho']:
    src = RUNS[name][1]
    if os.path.exists(src):
        shutil.copytree(src, os.path.join(stage, os.path.basename(src)))
for sub in ['pauc_figures', 'study_dsm_preproc', 'study_lrao']:
    s = os.path.join(RESULTS_DIR, sub)
    if os.path.exists(s):
        shutil.copytree(s, os.path.join(stage, sub))
for f in ['paper_2x2.pdf']:
    s = os.path.join(RESULTS_DIR, f)
    if os.path.exists(s): shutil.copy(s, stage)
shutil.make_archive(zip_base, 'zip', stage)
print('Bundle ->', zip_base + '.zip')

## 9. Architecture Ablation — Score-Model Variants

Compare **seven DSM score architectures** on IID single-class and multi-class detection.
All map x ∈ ℝᵈ → ℝᵈ and train with the **same** DSM loss + the **canonical additive
statistic** `t(y) = −(ψ(y)−ψ̄)ᵀs / √(sᵀC_ψ s)` (the minus-sign detector from
`make_pauc_figures.py`). Only ψ_η differs.

| Architecture | Description | #params (d=20) |
|---|---|---|
| `Linear` | Single linear layer d→d; ≈ AMF (Gaussian-optimal) | ~420 |
| `MLP-small` | Two-layer MLP [32, 32] | ~2 400 |
| `MLP (ours)` | Two-layer MLP [64, 64] — main paper model | ~6 800 |
| `MLP-big` | Two-layer MLP [128, 128] | ~21 800 |
| `Attn` | Squeeze-and-Excitation gate × linear proj | ~950 |
| `Conv1D` | 3× Conv1d (spectral dim as a length-d signal) | ~1 100 |
| `MoE` | Mixture of 3 linear experts, softmax gate | ~1 300 |

**Ablation axes** (sweep one, hold the rest at default):
- `n` = [50, 100, 200, 500, 1000, 2000, 4000]
- amplitude θ = [0.005, 0.01, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.7, 1.0]
- ρ = 15 pts, 0.0005→1.0 (denser near 0.01)
- d (latent / PCA dim)

**1000 epochs**, 3 seeds, CPU. **9a** = synthetic (Gaussian single-class; **GMM K=5,
d ≥ 20** multi-class, built so AMF/`Linear` break and only nonlinear scores detect).
**9b** = real Pavia-U.

> **Runtime / robustness.** These are large sweeps × 1000 epochs → **multi-hour** per cell
> on Colab CPU (the d=20 nets still lose on GPU to kernel-launch overhead). Each cell saves
> **incrementally after every axis** to its `.npz`, so a disconnect only loses the axis in
> progress — re-running resumes. Re-running a finished cell just reloads + replots.

In [ ]:
# ── 9.0  Architecture Zoo + Shared Helpers ────────────────────────────────────
import sys, os, time, warnings
import numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import roc_curve

sys.path.insert(0, PROJECT_DIR)
from dsm_model import ScoreNet, MixtureOfLinears, train_dsm

warnings.filterwarnings('ignore')
torch.set_num_threads(os.cpu_count() or 4)

# ── New architecture definitions (inline for ablation study) ──────────────────

class SpectralAttnNet(nn.Module):
    """
    Squeeze-and-Excitation score net: gate(x) * proj(x).
    gate = Sigmoid( Linear(r→d, SiLU(Linear(d→r, x))) )   [r = d//4]
    Lightweight attention-like mechanism over spectral bands.
    """
    def __init__(self, d, reduction=4):
        super().__init__()
        r = max(2, d // reduction)
        self.gate = nn.Sequential(nn.Linear(d, r), nn.SiLU(),
                                  nn.Linear(r, d), nn.Sigmoid())
        self.proj = nn.Linear(d, d)

    def forward(self, x):
        return self.gate(x) * self.proj(x)

    def n_params(self):
        return sum(p.numel() for p in self.parameters())


class Conv1DScoreNet(nn.Module):
    """
    1-D convolutional score net.
    Treats x ∈ R^d as a length-d, 1-channel spectral signal.
    Three Conv1d layers (1 → C → C → 1) with same-padding capture
    local spectral correlations while keeping param count small.
    """
    def __init__(self, d, channels=16, kernel=3):
        super().__init__()
        p = kernel // 2
        self.net = nn.Sequential(
            nn.Conv1d(1, channels, kernel, padding=p), nn.SiLU(),
            nn.Conv1d(channels, channels, kernel, padding=p), nn.SiLU(),
            nn.Conv1d(channels, 1, kernel, padding=p),
        )

    def forward(self, x):
        return self.net(x.unsqueeze(1)).squeeze(1)   # (B,d)→(B,1,d)→(B,d)

    def n_params(self):
        return sum(p.numel() for p in self.parameters())


# ── Architecture registry ─────────────────────────────────────────────────────
ARCH_NAMES = ['Linear', 'MLP-small', 'MLP (ours)', 'MLP-big', 'Attn', 'Conv1D', 'MoE']

# Consistent visual style (colour-blind-friendly palette)
ARCH_STYLES = {
    'Linear':     dict(color='#7f7f7f', ls='--', marker='x', lw=1.2),
    'MLP-small':  dict(color='#aec7e8', ls='-',  marker='o', lw=1.4),
    'MLP (ours)': dict(color='#d62728', ls='-',  marker='^', lw=2.2),  # highlighted
    'MLP-big':    dict(color='#ff7f0e', ls='-',  marker='s', lw=1.4),
    'Attn':       dict(color='#9467bd', ls='-',  marker='D', lw=1.4),
    'Conv1D':     dict(color='#2ca02c', ls='-',  marker='v', lw=1.4),
    'MoE':        dict(color='#1f77b4', ls='-',  marker='P', lw=1.4),
}

def make_arch(name, d):
    """Instantiate a score model by name for input dimension d."""
    if name == 'Linear':     return ScoreNet(d, [])
    if name == 'MLP-small':  return ScoreNet(d, [32, 32])
    if name == 'MLP (ours)': return ScoreNet(d, [64, 64])
    if name == 'MLP-big':    return ScoreNet(d, [128, 128])
    if name == 'Attn':       return SpectralAttnNet(d)
    if name == 'Conv1D':     return Conv1DScoreNet(d)
    if name == 'MoE':        return MixtureOfLinears(d, K=3, gate_hidden=8)
    raise ValueError(f'Unknown architecture: {name}')

# Print parameter counts
d_demo = 20
print(f"{'Architecture':<14}  {'#params':>8}")
print('─' * 28)
for nm in ARCH_NAMES:
    m = make_arch(nm, d_demo)
    print(f"  {nm:<14}  {sum(p.numel() for p in m.parameters()):>8,}")

# ── Training & scoring helpers ────────────────────────────────────────────────

def auto_sigma(bkg, rho):
    """σ = sqrt( ρ · tr(Σ̂) / d )  — paper's noise-level rule."""
    d   = bkg.shape[1]
    cov = np.cov(bkg.T) if bkg.shape[1] > 1 else np.var(bkg)
    tr  = float(np.trace(cov)) if np.ndim(cov) == 2 else float(cov)
    return float(np.sqrt(max(rho * tr / d, 1e-8)))


def fit_dsm(arch_name, bkg, sigma, d, epochs, lr=1e-3, wd=1e-4, bs=256, dev='cpu'):
    """Train a score model via DSM; return trained model on CPU."""
    model = make_arch(arch_name, d)
    return train_dsm(model, bkg, sigma,
                     lr=lr, batch_size=min(bs, len(bkg)),
                     epochs=epochs, device=dev,
                     print_every=epochs + 1,   # silent
                     weight_decay=wd)


def additive_scores(model, bkg_tr, X_te, s):
    """
    Additive DSM / LMP statistic — IDENTICAL to the paper pipeline's _score_dsm
    (experiments/honest_pipeline/make_pauc_figures.py, additive branch):

        t(y) = -(ψ(y) − ψ̄)ᵀ s / sqrt(sᵀ C_ψ s)

    The MINUS sign is the canonical detector: the additive log-likelihood is
    log p(y − θs), so the Rao/LMP score at θ=0 is dL/dθ = -sᵀ∇log p(y) = -sᵀψ(y),
    which is positive for targets (they score higher).  sqrt(sᵀ C_ψ s) is a
    positive per-set constant (does NOT change the ROC / Pd).  s is the FULL
    signature (not unit-normalised), matching the pipeline.
    """
    model.eval()
    with torch.no_grad():
        psi_b = model(torch.tensor(bkg_tr, dtype=torch.float32)).numpy()
        psi_t = model(torch.tensor(X_te,   dtype=torch.float32)).numpy()
    psi_mu = psi_b.mean(0)
    C      = np.cov(psi_b, rowvar=False)
    if C.ndim == 0:
        C = np.array([[float(C)]])
    norm = float(np.sqrt(max(float(s @ C @ s), 1e-12)))
    return -((psi_t - psi_mu) @ s) / norm      # (n_te,)  canonical minus sign


def pd_at_fa(scores, labels, fa=0.1):
    """P_det @ P_fa=fa via ROC interpolation."""
    fpr, tpr, _ = roc_curve(labels, scores)
    return float(np.interp(fa, fpr, tpr))


# ── Generic ablation runner ───────────────────────────────────────────────────

def run_ablation(data_fn, axis_name, axis_vals, default_params,
                 seeds, epochs, dev='cpu'):
    """
    Sweep one axis; for each value call
        bkg_tr, X_te, y_te, s = data_fn(seed=seed, **params)
    where params = {**default_params, axis_name: val}.
    Returns {arch_name: {val: [pd_seed0, pd_seed1, ...]}}.
    """
    results = {a: {} for a in ARCH_NAMES}
    total   = len(ARCH_NAMES) * len(axis_vals) * len(seeds)
    done    = 0
    t0      = time.time()

    for val in axis_vals:
        params = {**default_params, axis_name: val}
        n, d   = params['n'], params['d']
        for arch in ARCH_NAMES:
            pds = []
            for seed in seeds:
                bkg_tr, X_te, y_te, s = data_fn(seed=seed, **params)
                sigma  = auto_sigma(bkg_tr, params['rho'])
                model  = fit_dsm(arch, bkg_tr, sigma, d, epochs, dev=dev)
                scores = additive_scores(model, bkg_tr, X_te, s)
                pds.append(pd_at_fa(scores, y_te))
                done  += 1
            results[arch][val] = pds
        elapsed = time.time() - t0
        print(f"  {axis_name}={val!s:>8}  [{done:>4}/{total}]  {elapsed:5.0f}s", flush=True)

    return results


# ── Plot helper ───────────────────────────────────────────────────────────────

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 8,
    'axes.titlesize': 8, 'axes.labelsize': 8,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 150,
})

def plot_ablation_pair(ax_a, ax_b,
                       title_a, res_a,
                       title_b, res_b,
                       axis_vals, xlabel, log_x=False, fixed_str=''):
    for ax, (title, res) in zip([ax_a, ax_b],
                                 [(title_a, res_a), (title_b, res_b)]):
        for arch in ARCH_NAMES:
            st  = {k: v for k, v in ARCH_STYLES[arch].items()}
            lw  = st.pop('lw', 1.4)
            col = st['color']
            means = np.array([np.mean(res[arch][v]) for v in axis_vals])
            stds  = np.array([np.std( res[arch][v]) for v in axis_vals])
            ax.plot(axis_vals, means, label=arch, lw=lw, ms=4, **st)
            ax.fill_between(axis_vals, means - stds, means + stds,
                            color=col, alpha=0.12)
        ax.set_title(title, pad=3)
        ax.set_ylabel(r'$P_{\rm det}$ @ $P_{\rm fa}=0.1$')
        ax.set_xlabel(xlabel)
        ax.set_ylim(-0.02, 1.06)
        if log_x:
            ax.set_xscale('log')
        ax.yaxis.set_major_locator(mticker.MultipleLocator(0.2))
        ax.grid(True, ls=':', lw=0.4, alpha=0.6)
        if fixed_str:
            ax.text(0.98, 0.03, fixed_str, transform=ax.transAxes,
                    ha='right', va='bottom', fontsize=5.5, color='gray')

print('\nArch zoo + helpers ready.')


In [ ]:
# ── 9a. Toy Problem: IID single-class (Gaussian) + multi-class (GMM) ──────────
#
# Fully synthetic — no dataset needed.
#   Single-class  →  x ~ N(0, Σ)              (Gaussian, one mode)
#   Multi-class   →  x ~ GMM(K=5 well-separated modes, d>=20)
# Targets are PLANTED:  y = x_background + amp · s   (s = synthetic signature)
# Statistic = canonical additive DSM (minus sign), Metric = P_det @ P_fa=0.1.
#
# >>> The GMM is built so AMF / the linear score FAIL: the signature points along
# >>> the dominant BETWEEN-mode axis (pooled covariance inflated there), so a
# >>> global linear score is blind; only a nonlinear score detects. A printed AMF
# >>> sanity check confirms AMF≈chance on GMM but works on Gaussian.
#
# Sweep 4 axes (one at a time, others fixed at DEF_TOY): n · amp · rho · d
# Per-axis results are cached incrementally → safe across Colab disconnects.

SEEDS_TOY  = [42, 43, 44]
EPOCHS_TOY = 1000
DEV_TOY    = 'cpu'         # tiny nets: CPU beats GPU (kernel-launch overhead)
SIG_SCALE  = 2.5          # synthetic-signature magnitude (sets the amp→Pd range)

# Fixed defaults (held constant while sweeping each axis)
DEF_TOY = dict(n=500, amp=0.3, rho=0.01, d=20)

# (grid, x-label, log-x).  d-grid starts at 20 so the GMM stays in its (>=20) regime.
AXES_TOY = {
    'n':   ([50, 100, 200, 500, 1000, 2000, 4000],          'Training size $n$',   True),
    'amp': ([0.005, 0.01, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.7, 1.0],
                                                            'Amplitude $\\theta$', True),
    'rho': ([0.0005, 0.001, 0.002, 0.005, 0.008, 0.01, 0.015, 0.02, 0.03,
             0.05, 0.1, 0.2, 0.4, 0.7, 1.0],                'Noise level $\\rho$', True),
    'd':   ([20, 30, 40, 64],                               'Latent dim $d$',      False),
}

# ── Closed-form AMF (linear-score reference) — sanity check only ──────────────
def amf_scores(bkg_tr, X_te, s):
    mu    = bkg_tr.mean(0)
    Sigma = np.cov(bkg_tr.T) + 1e-6 * np.eye(bkg_tr.shape[1])
    w     = np.linalg.pinv(Sigma) @ s
    return (X_te - mu) @ w / np.sqrt(max(float(s @ w), 1e-12))

# ── Incremental, resumable multi-axis driver (used by 9a AND 9b) ──────────────
def run_all_axes(cache_path, axes, data_fns, keys, defaults, seeds, epochs, dev):
    """
    Run every axis for two datasets; save after each axis (disconnect-safe).
    A cached axis is REUSED only if its stored grid covers the requested values;
    if the grid changed (stale cache), that axis is recomputed automatically.
    """
    if os.path.exists(cache_path):
        z = dict(np.load(cache_path, allow_pickle=True))
        store = {k: (z[k].item() if k in z else {}) for k in keys}
        print('Resuming from cache:', cache_path)
    else:
        store = {k: {} for k in keys}

    def _axis_cached(ax_name, vals):
        for k in keys:
            d = store[k].get(ax_name)
            if not d:
                return False
            any_arch = next(iter(d.values()))                  # {val: [pds]}
            if not set(vals).issubset(set(any_arch.keys())):   # grid changed?
                return False
        return True

    for ax_name, (vals, _lab, _lx) in axes.items():
        if _axis_cached(ax_name, vals):
            print(f'  axis {ax_name}: cached, skip'); continue
        for k, fn in zip(keys, data_fns):
            print(f'\n── Sweeping {ax_name} ({k}) ──')
            store[k][ax_name] = run_ablation(fn, ax_name, vals, defaults, seeds, epochs, dev)
        np.savez_compressed(cache_path, **{k: store[k] for k in keys})
        print(f'  saved after axis {ax_name} -> {cache_path}')
    return tuple(store[k] for k in keys)

# ── Synthetic data generators ──────────────────────────────────────────────────
def _rand_cov(d, rng, scale=1.0):
    A = rng.standard_normal((d, d)) * scale
    return A @ A.T / d + np.eye(d) * 0.3

def gaussian_data(n, d, amp, rho, seed=42, test_n=2000):
    """IID single-class background ~ N(0, Σ). Additive planted target y = w + amp·s."""
    rng    = np.random.default_rng(seed)
    cov    = _rand_cov(d, rng)
    s      = rng.standard_normal(d); s = SIG_SCALE * s / np.linalg.norm(s)
    bkg_tr = rng.multivariate_normal(np.zeros(d), cov, n)
    bkg_te = rng.multivariate_normal(np.zeros(d), cov, test_n)
    n_tgt  = test_n // 2
    tgt_te = rng.multivariate_normal(np.zeros(d), cov, n_tgt) + amp * s
    X_te   = np.vstack([bkg_te, tgt_te])
    y_te   = np.array([0] * test_n + [1] * n_tgt)
    return bkg_tr, X_te, y_te, s

def gmm_data(n, d, amp, rho, n_comp=5, seed=42, test_n=2000, sep=6.0, wstd=0.35):
    """
    IID multi-class background ~ GMM with K=n_comp WELL-SEPARATED modes (d>=20).
    Built to break AMF: signature = top BETWEEN-mode axis (pooled cov inflated
    there → linear score blind); nonlinear score still detects the local shift.
    """
    rng     = np.random.default_rng(seed)
    means_k = np.stack([rng.standard_normal(d) for _ in range(n_comp)]) * sep
    covs_k  = [np.eye(d) * wstd**2 for _ in range(n_comp)]   # tight isotropic modes
    weights = rng.dirichlet(np.ones(n_comp) * 2.0)
    centered = means_k - means_k.mean(0)
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    s        = SIG_SCALE * vt[0] / np.linalg.norm(vt[0])      # between-mode signature
    def _sample(m):
        c = rng.multinomial(m, weights)
        return np.vstack([rng.multivariate_normal(means_k[k], covs_k[k], cc)
                          for k, cc in enumerate(c) if cc > 0])
    bkg_tr = _sample(n); bkg_te = _sample(test_n)
    n_tgt  = test_n // 2
    tgt_te = _sample(n_tgt) + amp * s
    X_te   = np.vstack([bkg_te, tgt_te])
    y_te   = np.array([0] * test_n + [1] * n_tgt)
    return bkg_tr, X_te, y_te, s

# ── Run (incremental) or load ─────────────────────────────────────────────────
CACHE_TOY = os.path.join(RESULTS_DIR, 'arch_ablation_toy.npz')

# sanity check: AMF must break on GMM but work on Gaussian
print('AMF (linear-score) sanity check at default config:')
for nm, fn in [('Gaussian', gaussian_data), ('GMM', gmm_data)]:
    bt, Xt, yt, sg = fn(seed=42, **DEF_TOY)
    flag = '   <- should be ~0.1 (broken)' if nm == 'GMM' else ''
    print(f'    AMF  Pd@0.1 on {nm:8s}: {pd_at_fa(amf_scores(bt, Xt, sg), yt):.3f}{flag}')

all_gauss, all_gmm = run_all_axes(
    CACHE_TOY, AXES_TOY, (gaussian_data, gmm_data), ('gauss', 'gmm'),
    DEF_TOY, SEEDS_TOY, EPOCHS_TOY, DEV_TOY)

# ── Plot: 4 rows × 2 columns ─────────────────────────────────────────────────
fig, axes = plt.subplots(4, 2, figsize=(9, 14))
fig.suptitle(
    'Architecture Ablation — Toy Synthetic Data\n'
    r'Metric: $P_{\rm det}$ @ $P_{\rm fa}=0.1$   (shaded = ±1 std over seeds)',
    fontsize=9, y=1.002)

for row, (ax_name, (ax_vals, ax_label, log_x)) in enumerate(AXES_TOY.items()):
    fixed = ', '.join(f'{k}={DEF_TOY[k]}' for k in ['n', 'amp', 'rho', 'd']
                      if k != ax_name)
    plot_ablation_pair(
        axes[row, 0], axes[row, 1],
        f'Gaussian (single-class)',           all_gauss[ax_name],
        f'GMM K=5, $d\\geq20$ (multi-class)',  all_gmm[ax_name],
        ax_vals, ax_label, log_x, fixed_str=fixed,
    )

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.012), fontsize=7, frameon=False)
fig.tight_layout(rect=[0, 0.025, 1, 1])

out_toy = os.path.join(RESULTS_DIR, 'arch_ablation_toy.pdf')
fig.savefig(out_toy, bbox_inches='tight')
print('Saved:', out_toy)
plt.show()


In [ ]:
# ── 9b. Real Pavia-U: Architecture Ablation on HSI Data ───────────────────────
#
# Real Pavia University HSI (same dataset as the main paper). Two settings:
#   Single-class  bkg=class 2 (meadows),      tgt=class 1 (asphalt)
#   Multi-class   bkg=all labeled non-target,  tgt=class 1 (asphalt)
#
# Honest IID protocol:
#   1. Per-band standardise the full image.
#   2. Fit PCA on background pixels → reduce to d dims  (CACHED per (bkg,d)).
#   3. Signature s = mean(tgt_pca) - mean(bkg_pca)   [FULL signature, not unit].
#   4. Plant additive targets:  y = w + amp·s   (matches the pipeline / paper).
#   5. Train DSM (canonical minus-sign statistic); metric P_det @ P_fa=0.1.
#
# Sweep axes (your grids): n · amp · rho · d. 1000 epochs. Incremental cache.

import scipy.io
from sklearn.decomposition import PCA

SEEDS_PAVIA  = [42, 43, 44]
EPOCHS_PAVIA = 1000
DEV_PAVIA    = 'cpu'        # tiny nets: CPU beats GPU (kernel-launch overhead)

# ── Load + per-band standardise Pavia ────────────────────────────────────────
_mat      = scipy.io.loadmat(DATA)
_img      = _mat['data'].astype(np.float32)   # (H, W, 103)
_gt       = _mat['map'].astype(int).ravel()    # (H*W,)
_flat_raw = _img.reshape(-1, _img.shape[2])
_mu_b     = _flat_raw.mean(0); _sd_b = _flat_raw.std(0).clip(1e-8)
PAVIA_FLAT = (_flat_raw - _mu_b) / _sd_b      # per-band standardised
PAVIA_GT   = _gt

print(f'Pavia: {PAVIA_FLAT.shape[0]:,} pixels, {PAVIA_FLAT.shape[1]} bands')
_cls, _cnt = np.unique(PAVIA_GT[PAVIA_GT > 0], return_counts=True)
for c, k in zip(_cls, _cnt):
    print(f'  class {c}: {k:>6,} pixels')

SINGLE_BKG = 2   # meadows
MULTI_BKG  = [c for c in _cls if c != 1]
TGT_CLS    = 1   # asphalt

# ── PCA cache: fit once per (bkg_cls, d), reuse across seeds / sweep values ────
_PCA_CACHE = {}
def _pavia_pca(bkg_cls, d):
    key = (str(bkg_cls), int(d))
    if key not in _PCA_CACHE:
        bkg_mask = (PAVIA_GT == bkg_cls) if isinstance(bkg_cls, int) \
                   else np.isin(PAVIA_GT, bkg_cls)
        bkg_raw = PAVIA_FLAT[bkg_mask]
        tgt_raw = PAVIA_FLAT[PAVIA_GT == TGT_CLS]
        pca     = PCA(n_components=d, random_state=42)
        bkg_pca = pca.fit_transform(bkg_raw)
        tgt_pca = pca.transform(tgt_raw)
        _PCA_CACHE[key] = (bkg_pca, tgt_pca)
    return _PCA_CACHE[key]

# ── Pavia data function ───────────────────────────────────────────────────────
def pavia_data(bkg_cls, n, d, amp, rho, seed=42, test_n_bkg=1000, test_n_tgt=500):
    """Honest IID protocol on Pavia-U. Plants the FULL signature: y = w + amp·s."""
    rng = np.random.default_rng(seed)
    bkg_pca, tgt_pca = _pavia_pca(bkg_cls, d)          # cached PCA
    s = tgt_pca.mean(0) - bkg_pca.mean(0)              # FULL signature (not unit)

    n_avail = len(bkg_pca)
    tr_idx  = rng.choice(n_avail, size=min(n, n_avail), replace=False)
    bkg_tr  = bkg_pca[tr_idx]
    te_bkg_idx = rng.choice(n_avail, size=min(test_n_bkg, n_avail), replace=False)
    te_tgt_idx = rng.choice(n_avail, size=min(test_n_tgt, n_avail), replace=False)
    bkg_te  = bkg_pca[te_bkg_idx]
    tgt_te  = bkg_pca[te_tgt_idx] + amp * s            # plant amp · FULL signature

    X_te = np.vstack([bkg_te, tgt_te])
    y_te = np.array([0] * len(bkg_te) + [1] * len(tgt_te))
    return bkg_tr, X_te, y_te, s

def pavia_single(n, d, amp, rho, seed=42):
    return pavia_data(SINGLE_BKG, n, d, amp, rho, seed)

def pavia_multi(n, d, amp, rho, seed=42):
    return pavia_data(MULTI_BKG,  n, d, amp, rho, seed)

# Defaults (n=2000 / amp=0.2 so the multi-class panels are informative at fixed axes)
DEF_PAVIA = dict(n=2000, amp=0.2, rho=0.01, d=20)

# Axis grids (your spec; d includes low dims since real PCA dims are meaningful)
AXES_PAVIA = {
    'n':   ([50, 100, 200, 500, 1000, 2000, 4000],          'Training size $n$',   True),
    'amp': ([0.005, 0.01, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.7, 1.0],
                                                            'Amplitude $\\theta$', True),
    'rho': ([0.0005, 0.001, 0.002, 0.005, 0.008, 0.01, 0.015, 0.02, 0.03,
             0.05, 0.1, 0.2, 0.4, 0.7, 1.0],                'Noise level $\\rho$', True),
    'd':   ([5, 10, 20, 40, 64],                            'PCA dim $d$',         False),
}

# ── Run (incremental) or load — reuses run_all_axes defined in cell 9a ────────
CACHE_PAVIA = os.path.join(RESULTS_DIR, 'arch_ablation_pavia.npz')
all_sg, all_mu = run_all_axes(
    CACHE_PAVIA, AXES_PAVIA, (pavia_single, pavia_multi), ('single', 'multi'),
    DEF_PAVIA, SEEDS_PAVIA, EPOCHS_PAVIA, DEV_PAVIA)

# ── Plot: 4 rows × 2 columns ─────────────────────────────────────────────────
fig, axes = plt.subplots(4, 2, figsize=(9, 14))
fig.suptitle(
    'Architecture Ablation — Real Pavia-U HSI\n'
    r'Metric: $P_{\rm det}$ @ $P_{\rm fa}=0.1$   (shaded = ±1 std over seeds)',
    fontsize=9, y=1.002)

for row, (ax_name, (ax_vals, ax_label, log_x)) in enumerate(AXES_PAVIA.items()):
    fixed = ', '.join(f'{k}={DEF_PAVIA[k]}' for k in ['n', 'amp', 'rho', 'd']
                      if k != ax_name)
    plot_ablation_pair(
        axes[row, 0], axes[row, 1],
        'Single-class  (bkg=meadows)',  all_sg[ax_name],
        'Multi-class   (bkg=all)',      all_mu[ax_name],
        ax_vals, ax_label, log_x, fixed_str=fixed,
    )

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.012), fontsize=7, frameon=False)
fig.tight_layout(rect=[0, 0.025, 1, 1])

out_pavia = os.path.join(RESULTS_DIR, 'arch_ablation_pavia.pdf')
fig.savefig(out_pavia, bbox_inches='tight')
print('Saved:', out_pavia)
plt.show()
